# NLP Feature Extraction
- Part-of-Speech Tagging
- Term Frequency-Inverse Document Frequency
- Sentiment Analysis
- Pretrained Word Embeddings
- Custom Word Embbeding Model
- Speaker / Speaker history

## Import Dependencies

In [1]:
import spacy
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from typing import List
from utils import load_model_and_tokenizer, load_inference_data
from sentiment_analysis_eval import get_predictions
import gzip
from gensim.models import Word2Vec

c:\Users\nickb\OneDrive\Documenten\GitHub\fae2-nlpr-group-group-4-1\nlp_cia\.venv\Lib\site-packages\moviepy\config_defaults.py:1: SyntaxWarning: invalid escape sequence '\P'
  """
c:\Users\nickb\OneDrive\Documenten\GitHub\fae2-nlpr-group-group-4-1\nlp_cia\.venv\Lib\site-packages\moviepy\video\io\ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
c:\Users\nickb\OneDrive\Documenten\GitHub\fae2-nlpr-group-group-4-1\nlp_cia\.venv\Lib\site-packages\moviepy\video\io\ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
c:\Users\nickb\OneDrive\Documenten\GitHub\fae2-nlpr-group-group-4-1\nlp_cia\.venv\Lib\site-packages\moviepy\video\io\ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)


# Helper Functions


In [2]:
def load_df(filepath: str) -> pd.DataFrame:
    """Function to load a pd.DataFrame from a given filepath"""
    try:
        if filepath.endswith('.csv'):
            df = pd.read_csv(filepath)
        elif filepath.endswith('.xlsx'):
            df = pd.read_excel(filepath)
        else:
            raise ValueError("Unsupported file format. Please use .csv or .xlsx")
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {filepath}")
    return df

def load_sentences(filepath: str, text_column: str = "Sentence") -> pd.Series:
    """Function to load a column from a pd.DataFrame into a list
    Author: Nick Belterman"""
    df = load_df(filepath)
    return df[text_column]


## Create Variables

In [3]:
language_code_mapping = {
    'nl': 'Dutch',
    'en': 'English',
    'de': 'German',
    'fr': 'French',
    'es': 'Spanish',
    'it': 'Italian',
    'pt': 'Portuguese',
    'ru': 'Russian',
    'zh': 'Chinese',
    'ja': 'Japanese',
    'ko': 'Korean',
    'ar': 'Arabic',
    'hi': 'Hindi',
    'bn': 'Bengali',
    'ur': 'Urdu',
    'vi': 'Vietnamese',
    'tr': 'Turkish',
    'sv': 'Swedish',
    'da': 'Danish',
    'no': 'Norwegian',
    'fi': 'Finnish',
    'pl': 'Polish',
    'cs': 'Czech',
    'hu': 'Hungarian',
    'ro': 'Romanian',
    'el': 'Greek',
    'he': 'Hebrew',
}

language = 'nl'  # Change this to the desired language code
transcription_file = "..\\data\\transcription\\csv\\transcription.csv"  # Change this to your transcription file path
output_file = '..\\data\\features\\NLP_features_output.csv'  # Change this to your desired output file path
print(f"Download model using `poetry run python -m spacy download {language}_core_news_lg` if not already installed.")

Download model using `poetry run python -m spacy download nl_core_news_lg` if not already installed.


## Function to Load Spacy Model

In [4]:

def load_spacy_model(
    language: str = 'nl', 
    text_column: str = 'Sentence',
    ) -> spacy.language.Language:
    """Load the spaCy model for the specified language.
    Args:
        language (str): Language code (default is 'nl' for Dutch).
    Returns:
        spacy.language.Language: Loaded spaCy language model.
    Raises:
        ValueError: If the language code is not supported.
    Authors: Floris Lokhorst, Nick Belterman
    """
    model_name = f"{language}_core_news_lg"
    try:
        language_name = language_code_mapping.get(language)
    except KeyError:
        raise ValueError(f"Unsupported language code: {language}")

    # Configuration
    CONFIG = {
        'spacy_model': model_name,
        'language': language_name,
        'text_column': text_column,  
    }

    return spacy.load(CONFIG['spacy_model']), CONFIG

## Functions to Extract POS tags

In [5]:
def extract_save_pos_tags(
    input_filepath: str, 
    nlp_model: spacy.language.Language, \
    output_filepath: str = '..\\data\\features\\NLP_features_output.csv',
    config: dict = None
) -> None:
    """Extract POS tags from text.
    
    Authors: Floris Lokhorst, Nick Belterman
    """
    df  = load_df(input_filepath)
    text_column = config.get('text_column', 'Sentence') if config else 'Sentence'
    if not os.path.exists(os.path.dirname(output_filepath)):
        os.makedirs(os.path.dirname(output_filepath))
    data = load_sentences(input_filepath, text_column=text_column)
    # Process each sentence and store the POS tags
    df['POS_tags'] = data.astype(str).apply(
        lambda text: ' '.join(f"{token.text}_{token.pos_}" for token in nlp_model(text.strip())) if text and text.strip() else ''
    )
    
    # Placeholders for start_time and end_time
    data['start_time'] = np.nan
    data['end_time'] = np.nan

    # Save the DataFrame to a CSV file
    try:
        df.to_csv(output_filepath, index=False)
        print(f"Successfully processed {len(data)} sentences and saved the output to {output_filepath}")
    except Exception as e:
        print(f"Error saving the CSV file: {e}")

# Code to extract and save POS tags using spacy

In [8]:
nlp, config = load_spacy_model()
extract_save_pos_tags(transcription_file, nlp, output_file, config)


Successfully processed 48 sentences and saved the output to ..\data\features\NLP_features_output.csv


# TF-IDF

# Function to load and fit TF-IDF Vectorizer on our chosen episode 

In [12]:
def load_fit_tfidf(
    sentences: List[str], 
) -> (np.ndarray, List[str]):
    """Load and fit a TF-IDF vectorizer on the provided sentences.
    
    Args:
        sentences (List[str]): List of sentences to fit the TF-IDF vectorizer.
    
    Authors: Jeroen Brower, Nick Belterman
    """
    vectorizer = TfidfVectorizer(
        max_features=2000,         
        min_df=1,                    
        max_df=1.0,                  
        lowercase=True,
        stop_words=None,             
        token_pattern=r'\b[a-záéíóúàèìòùâêîôûäëïöüç]+\b',  
    )
    print("Fitting TF-IDF...")
    tfidf_matrix = vectorizer.fit_transform(sentences)
    vocab = vectorizer.get_feature_names_out()
    print(f"Total vocabulary: {len(vocab)}")
    return tfidf_matrix, vocab

# Function to add the TF-IDF matrix to our feature extraction output file

In [13]:
def create_tfidf_dataframe(
    tfidf_matrix: np.ndarray, 
    vocab: List[str],
    output_filepath: str = '..\\data\\features\\NLP_features_output.csv',
    text_column: str = 'Sentence',
    readable: bool = False,
) -> pd.DataFrame:
    """Create a DataFrame from the TF-IDF matrix and vocabulary.
    
    Args:
        tfidf_matrix (np.ndarray): The TF-IDF matrix.
        vocab (List[str]): The vocabulary list.
    
    Returns:
        pd.DataFrame: DataFrame with TF-IDF features.
    
    Authors: Jeroen Brower, Nick Belterman
    """
    df = load_df(transcription_file)
    sentences = load_sentences(transcription_file, text_column=text_column)

    if readable:
        print("Creating readable TF-IDF format...")
        # Create readable format
        tfidf_readable = []
        for i in range(len(sentences)):
            vector = tfidf_matrix[i].toarray()[0]
            
            # Get ALL words with their scores
            all_scores = [(vocab[j], score) for j, score in enumerate(vector) if score > 0]
            all_scores.sort(key=lambda x: x[1], reverse=True)
            
            # Format top words
            if all_scores:
                formatted = [f"{word}({score:.3f})" for word, score in all_scores[:8]]
                result = "; ".join(formatted)
            else:
                result = "EMPTY"
            
            tfidf_readable.append(result)
        df['TF_IDF'] = tfidf_readable
    else:
        print("Creating non-readable TF-IDF format...")
        # Non-readable format (sparse)
        tfidf_list = []
        for i in range(len(sentences)):
            vector = tfidf_matrix[i].toarray()[0]
            indices = [j for j, score in enumerate(vector) if score > 0]
            scores = [vector[j] for j in indices]
            pairs = [f"{vocab[j]}({score:.3f})" for j, score in zip(indices, scores)]
            result = "; ".join(pairs) if pairs else "EMPTY"
            tfidf_list.append(result)
        df['TF_IDF'] = tfidf_list


    # Dropna
    df.dropna(subset=[text_column], inplace=True)
    # Add to dataframe
    
    df.to_csv(output_filepath, index=False)
    print(f"Saved TF-IDF features to {output_filepath}")

# Code to extract and save TF-IDF terms using Scikit-learn

In [14]:
sentences = load_sentences(output_file, text_column='Sentence').dropna()
tfidf_matrix, vocab = load_fit_tfidf(sentences)
create_tfidf_dataframe(tfidf_matrix, vocab, output_file)

Fitting TF-IDF...
Total vocabulary: 138
Creating non-readable TF-IDF format...
Saved TF-IDF features to ..\data\features\NLP_features_output.csv


In [15]:
df = pd.read_csv(output_file)
print(df['TF_IDF'][0])

ah(0.537); de(0.329); maartje(0.418); ouders(0.537); van(0.375)


# Sentiment Analysis

Model can be found in [Models_nlp_2526](https://edubuas-my.sharepoint.com/:f:/g/personal/225538_buas_nl/EuvOvnKbcLdPnsq8LGqDbJ0BP0DLEER7PMAwyS1bOy4JZg?e=olbDPM) and should be saved in `nlp_cia\\models\\` 

In [16]:
df = df.dropna(subset=['Sentence'])
# subset = df.iloc[:15]
tokenizer, model = load_model_and_tokenizer(model_dir=r"..\\models\\Model_14e03c00")
sentences, df = load_inference_data(tokenizer, inference_data=df)
preds = get_predictions(model, sentences)
print(f"Predictions: {preds}")
df.to_csv(output_file, index=False)

Loading model and tokenizer from: ..\\models\\Model_14e03c00
c:\Users\nickb\OneDrive\Documenten\GitHub\fae2-nlpr-group-group-4-1\nlp_cia\src
texts = ['Ah, de ouders van Maartje', 'Ja, hallo', 'Hallo', 'Zo', 'Nou', 'Ja, wat fijn dat jullie allebei naar dit tienminutengesprek konden komen', 'Waar zou ik beginnen', 'Is er iets', 'Heeft ze zich misdragen', 'Zijn de cijfers onvoldoende', 'Nou, Maartje tekent nogal graag, h', 'O, ja, ja, ja', 'Ze is een heel creatief meisje', 'Ja, dat heeft ze van mij', 'Maar is het te veel', 'Moet ze minder tekenen', 'Want dan ga ik dat tegen haar zeggen', 'Ja, ik was vroeger ook altijd bij potten', 'Gewoon, hij stift in de weer de hele dag', 'Nou, het gaat ons meer om wat ze tekent', 'Laatst was de opdracht teken je lievelingsdier', 'En dan komt Maartje met dit', 'Tja, het is op zich wel goed getekend', 'Ja, Miepie is ook goed geschreven', 'Vorige week was de opdracht teken een zelfportret', 'Daar ziet u wat voorbeelden van de andere kinderen', 'Leuk', 'Ja

## Custom Word Embeddings
### We used the dutch OpenSubtitles Corpus as it matches our goal for TV emotion classification

OpenSubtitles Dutch Corpus (OPUS Collection)
Source: Movie and TV subtitle translations from OpenSubtitles.org, compiled by the OPUS project NlplPapers with Code

https://opus.nlpl.eu/OpenSubtitles/nl&en/v2024/OpenSubtitles

https://www.opensubtitles.org/en/search/subs


Scale: Part of collection spanning 60 corpora in 58 languages with 2.6 billion sentences total OpenSubtitles parallel corpora

Domain: Movies and television shows (conversational dialogue)

Language variety:  Dutch

Text type: Subtitle transcripts representing natural dialogue and speech patterns

Countries of origin: Dutch-language films and international content dubbed/subtitled in Dutch

Temporal range: Various decades of film/TV content up to recent releases

Data structure: Monolingual Dutch sentences extracted from subtitle files



## Code to extract relevant information from OpenSubtitles dataset

In [17]:
def extract_and_prepare_corpus(gz_file_path, max_sentences=1_000_000):
    """Extract corpus, show statistics, and prepare for training"""
    
    # Extract the file
    output_file = gz_file_path.replace('.gz', '')
    
    with gzip.open(gz_file_path, 'rt', encoding='utf-8') as f_in:
        with open(output_file, 'w', encoding='utf-8') as f_out:
            f_out.write(f_in.read())
    
    sentences = []
    line_lengths = []
    
    print("First 10 lines:")
    with open(output_file, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f):
            line = line.strip()
            
            # Show first 10 lines
            if line_num < 10:
                print(f"{line_num+1}: {line}")
            
            if line:
                words = line.split()
                if len(words) >= 1:
                    sentences.append(words)
                    line_lengths.append(len(words))
                
                # Stop at max for training
                if len(sentences) >= max_sentences:
                    break
    
    # Show statistics
    total_processed = line_num + 1
    print(f"\nDataset Statistics:")
    print(f"Total lines processed: {total_processed:,}")
    print(f"Sentences for training: {len(sentences):,}")
    print(f"Average tokens per sentence: {sum(line_lengths)/len(line_lengths):.1f}")
    print(f"Min/Max tokens: {min(line_lengths)}/{max(line_lengths)}")
    
    return sentences, output_file

In [18]:
training_sentences, corpus_file = extract_and_prepare_corpus("nl.tok.gz")

FileNotFoundError: [Errno 2] No such file or directory: 'nl.tok.gz'

In [19]:
def train_and_evaluate_model(sentences, params, name):
    """Train model with specific parameters and evaluate"""
    print(f"\n=== Training {name} ===")
    print(f"Parameters: {params}")
    
    model = Word2Vec(sentences=sentences, **params)
    
    # Quick evaluation
    test_words = ['goed', 'slecht', 'mooi', 'blij']
    print(f"Vocabulary size: {len(model.wv.index_to_key)}")
    
    for word in test_words:
        if word in model.wv:
            similar = model.wv.most_similar(word, topn=3)
            print(f"'{word}': {[w for w, score in similar]}")
    
    return model

# Test different configurations
configs = {
    "Baseline": {"vector_size": 200, "window": 5, "min_count": 5, "sg": 1, "epochs": 10, "workers": 8},
    "larger_vector_size": {"vector_size": 500, "window": 5, "min_count": 5, "sg": 1, "epochs": 10, "workers": 8},
    "smaller_window": {"vector_size": 200, "window": 3, "min_count": 5, "sg": 1, "epochs": 10, "workers": 8},
    "higher_mincount": {"vector_size": 200, "window": 5, "min_count": 10, "sg": 1, "epochs": 10, "workers": 8},
    "cbow_model": {"vector_size": 200, "window": 5, "min_count": 5, "sg": 0, "epochs": 10, "workers": 8},
    "Higher_epochs": {"vector_size": 200, "window": 5, "min_count": 5, "sg": 0, "epochs": 20, "workers": 8}
}

models = {}
for name, params in configs.items():
    models[name] = train_and_evaluate_model(training_sentences, params, name)

NameError: name 'training_sentences' is not defined

Parameter Justification:
- vector_size = 500: Balance between expressiveness and efficiency and no longer had the goed/slecht pairing
- window = 5: Captures sufficient conversational context
- min_count = 5: Filters noise, reduces vocab from 52K to 30K words
- sg = 1 (Skip-gram): Better for rare emotional vocabulary than CBOW
- epochs = 10: Sufficient convergence without overfitting

In [20]:
print(f"Using {len(training_sentences)} sentences for training")
model = Word2Vec(
    sentences=training_sentences,
    vector_size=500,
    window=5,
    min_count=5,
    workers=4,
    epochs=10,
    sg=1,
    seed=42
)

model.save("dutch_custom_embeddings_final.model")

print(f"Training completed with {len(model.wv.index_to_key)} vocabulary words")

NameError: name 'training_sentences' is not defined

### Observed Results:
- Successfully learned semantic clusters (e.g., "mooi" with "prachtig", "fraai")
- Identified challenge: antonyms co-occur in dialogue


## Pretrained Word Embeddings

In [24]:
print(os.getcwd())
df = pd.read_csv(r"../data/wie_is_de_mol_sentiment.csv")

nlp = spacy.load("nl_core_news_lg")
vec_size = nlp.vocab.vectors_length

# Function to compute sentence embeddings
def sent_embedding_spacy(text: str) -> np.ndarray:
    doc = nlp(text)
    vecs = [t.vector for t in doc if t.has_vector]
    return np.mean(vecs, axis=0) if vecs else np.zeros(vec_size, dtype=np.float32)

# Compute embeddings for each row in the 'text' column
df['embedding'] = df['sentence'].apply(sent_embedding_spacy)
embeddings_array = np.stack(df['embedding'].values)
print(embeddings_array.shape)

c:\Users\nickb\OneDrive\Documenten\GitHub\fae2-nlpr-group-group-4-1\nlp_cia\src
(1083, 300)
